# Markovian Mpemba effect

*thermodynamics · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/thermodynamics/mpemba_effect)

Can hot water freeze faster than warm water? For a particle hopping in a lattice potential with Metropolis rates the answer is yes. The master equation $\dot p = W p$ is diagonalised once, every initial temperature is projected on the slowest mode in one einsum, and a hot start whose slow-mode amplitude $a_2$ vanishes relaxes exponentially faster than a cooler one.

**Vector form**

$$\dot{\mathbf p} = W\,\mathbf p,\qquad \mathbf p(t) = \boldsymbol\pi + \sum_{k\ge 2} a_k\,e^{\lambda_k t}\,\mathbf r_k,\qquad a_k = \boldsymbol\ell_k\cdot\mathbf p(0)$$

**Index (tensor) form**

$$W_{ij} = A_{ij}\min\!\big(1, e^{-(U_i - U_j)/T_b}\big) - \delta_{ij}\textstyle\sum_k A_{kj}\min\!\big(1, e^{-(U_k - U_j)/T_b}\big),\qquad \dot p_{bi} = W_{ij}\,p_{bj},\qquad a_{bk} = \ell_{ik}\,p_{bi}(0)$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $U_i$ | `potential(barrier, dev)` | (N,) | staircase energy of site $i$: plateau, deep narrow well, barrier, wide shallow well |
| $A_{ij}$ | `neighbours(N, dev)` | (N, N) | 1 for nearest neighbours on the chain, else 0 |
| $W_{ij}$ | `metropolis_rates(u, A, T_bath)` | (N, N) | rate of a jump $j \to i$; the diagonal makes every column sum to zero |
| $p_{bi}(0)$ | `softmax(-u / T0[:, None], dim=1)` | (B, N) | Boltzmann state at each initial temperature $T_{0,b}$, a batch of starts |
| $W_{ij}\,p_{bj}$ | `einsum('ij,bj->bi', W, p)` | (B, N) | repeated $j$ ⇒ summed: the master equation for every start at once |
| $\lambda_k,\ r_{ik},\ \ell_{ik}$ | `spectrum(W, pi)` | (N,), (N, N), (N, N) | eigh of the symmetrised $\Pi^{-1/2} W \Pi^{1/2}$; $\lambda_1 = 0$ is equilibrium, $\lambda_2$ the slowest decay |
| $a_{bk} = \ell_{ik}\,p_{bi}(0)$ | `einsum('ik,bi->bk', left, p0)` | (B, N) | repeated $i$ ⇒ summed: overlap of each start with each mode |
| $r_{ik}\,a_{bk}\,e^{\lambda_k t}$ | `einsum('ik,bk,tk->tbi', right, a, decay)` | (T, B, N) | repeated $k$ ⇒ summed: exact solution at every time |
| $D(p\,\Vert\,\pi)$ | `kl_divergence(p, pi)` | (T, B) | Kullback–Leibler distance to equilibrium, $\sum_i p_i \ln(p_i/\pi_i)$ |

Units: $k_B = 1$ and the attempt rate is 1. The bath sits at $T_b$ and the system starts in equilibrium at a temperature $T_0$. If the hops obey detailed balance, $W_{ij}\pi_j = W_{ji}\pi_i$, $W$ is similar to a symmetric matrix, so all its eigenvalues are real and $\le 0$, and its stationary state is the Boltzmann distribution $\pi$. After a short time only the slowest mode is left, $p - \pi \approx a_2\, e^{\lambda_2 t}\, r_2$, so the start with the smaller $\lvert a_2\rvert$ ends up closer to equilibrium, whatever its temperature. Because $a_2(T_0)$ need not be monotonic, a hotter start can overtake a warmer one (the Markovian Mpemba effect). At the special $T^*$ where $a_2 = 0$ the slow mode is absent altogether, and the relaxation runs at the faster rate $\lambda_3$ (the strong effect). In this potential the shape comes from a competition between basins. At low $T_0$ the deep narrow well holds most of the probability. At intermediate $T_0$ the wide shallow well wins by entropy. At high $T_0$ the raised plateau fills, so the left basin's share comes back up and passes its equilibrium value at $T^*$. The theory is from Lu & Raz (PNAS, 2017). Kumar & Bechhoefer (Nature, 2020) saw the effect, including the strong version, with a colloidal bead in an optically shaped double-well potential. Kumar, Chétrite & Bechhoefer (PNAS, 2022) then saw the inverse effect (anomalous heating). Quantum analogues have recently been reported in trapped-ion experiments. This lattice is a toy version of those systems. Try raising $T_b$: above about $1.4$ the crossing disappears.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/thermodynamics/mpemba_effect/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import evolve, kl_divergence, mode_amplitudes


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    T_bath, T_hot = params["T_bath"], params["T_hot"]
    m = solve.build(params["barrier"], T_bath, dev)  # u, W, π, λ, r, ℓ
    u, lam = m["u"], m["lam"]
    print(f"{len(u)} sites, bath T_b = {T_bath:.2f}")
    print(f"columns of W sum to {m['W'].sum(dim=0).abs().max():.1e};  λ_1 = {lam[0]:.1e}")
    print(f"slowest relaxation λ_2 = {lam[1]:.3e} (τ_2 = {-1 / lam[1]:.0f})")
    print(f"next mode          λ_3 = {lam[2]:.3e} (τ_3 = {-1 / lam[2]:.0f})")

    # a_2(T) on a fine log grid: one batched einsum over 300 temperatures
    T_grid = torch.logspace(math.log10(0.8 * T_bath), math.log10(100 * T_bath), 300, dtype=u.dtype, device=dev)
    a2_grid = solve.a2_curve(u, T_grid, m["left"])
    T_star = solve.mpemba_point(u, T_bath, m["left"])
    print(f"strong Mpemba point: a_2(T*) = 0 at T* = {T_star:.3f}")

    # the batch of starts: log-spaced temperatures, the Mpemba point, the warmest |a_2| below T*, and T_hot
    between = (T_grid > T_bath) & (T_grid < T_star)
    T_warm = T_grid[between][a2_grid[between].abs().argmax()].item() if between.any() else 1.5 * T_bath
    T0 = tensors.start_temperatures(T_bath, T_star, 9, dev)
    T0 = torch.cat([T0, torch.tensor([T_warm, T_hot], dtype=u.dtype, device=dev)]).unique()  # sorted (B,)
    p0 = tensors.boltzmann(u, T0)  # (B, N)
    a = mode_amplitudes(p0, m["left"])  # (B, N)

    times = torch.cat([torch.zeros(1), torch.logspace(-1, math.log10(8 / -lam[1].item()), 299)]).to(u)
    p = evolve(a, lam, m["right"], times)  # (T, B, N)
    kl = kl_divergence(p, m["pi"])  # (T, B)

    show = {k: int((T0 - v).abs().argmin()) for k, v in {"warm": T_warm, "T*": T_star, "hot": T_hot}.items()}
    for name, b in show.items():
        print(f"{name:>4}: T0 = {T0[b]:6.3f}  a_2 = {a[b, 1]:+.4f}  D(p‖π) = {kl[0, b]:.3f} → {kl[-1, b]:.1e}")
    w, h = show["warm"], show["hot"]
    if abs(a[h, 1]) < abs(a[w, 1]):
        cross = (kl[:, h] < kl[:, w]) & (times > 0)
        t_x = times[cross.nonzero()[0, 0]].item() if cross.any() else float("nan")
        print(f"Mpemba effect: the T0 = {T0[h]:.2f} start overtakes the T0 = {T0[w]:.2f} start at t ≈ {t_x:.0f}")
    else:
        print(f"no Mpemba effect between T0 = {T0[w]:.2f} and T0 = {T0[h]:.2f}: |a_2| is larger for the hotter start")
    return plot.animate(
        u, m["pi"], times, p, kl, T0, show, T_grid, a2_grid, T_bath, T_star, out_dir / "mpemba_effect.gif"
    )


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/thermodynamics/mpemba_effect/equation.py
r"""Physics → tensor translation.

A particle hops between neighbouring sites i of a 1D lattice with energies U_i, kicked by a bath at
temperature T_b. The probability vector p_i(t) obeys the master equation

    \dot p_i = W_{ij} p_j                                    (j summed)

with Metropolis rates for a jump j → i between neighbours (A_ij = 1):

    W_{ij} = A_{ij} \min(1, e^{-(U_i - U_j)/T_b})   (i ≠ j),        W_{jj} = -\sum_{k} W_{kj}

The diagonal makes every column sum to zero, so total probability is conserved. The rates obey detailed
balance, W_ij π_j = W_ji π_i, so the stationary state is the Boltzmann distribution π_i ∝ e^{-U_i/T_b}.

Expanding in the right/left eigenvectors of W (W r_k = λ_k r_k, ℓ_k^T W = λ_k ℓ_k^T):

    p_{bi}(t) = π_i + \sum_{k ≥ 2} a_{bk} e^{λ_k t} r_{ik},       a_{bk} = ℓ_{ik} p_{bi}(0)     (i summed)

b labels the batch of initial temperatures. At long times only the slowest mode k = 2 survives, so the
smaller |a_2|, the closer to equilibrium. A hot start can beat a warm one (the Mpemba effect).
"""

import torch


def metropolis_rates(u: torch.Tensor, A: torch.Tensor, T_bath: float) -> torch.Tensor:
    r"""W_{ij} = A_{ij} min(1, e^{-(U_i-U_j)/T_b}) off the diagonal, columns summing to zero.   → (N, N)"""
    uphill = u[:, None] - u[None, :]  # (N, N)  U_i - U_j: energy gained jumping j → i
    W = A * torch.clamp(torch.exp(-uphill / T_bath), max=1.0)  # (N, N)
    return W - torch.diag(W.sum(dim=0))  # W_jj = -Σ_k W_kj  (escape rate out of j)


def master_rhs(p: torch.Tensor, W: torch.Tensor) -> torch.Tensor:
    r"""\dot p_{bi} = W_{ij} p_{bj}   (j summed).   p: (B, N) → (B, N)"""
    return torch.einsum("ij,bj->bi", W, p)


def mode_amplitudes(p0: torch.Tensor, left: torch.Tensor) -> torch.Tensor:
    r"""a_{bk} = ℓ_{ik} p_{bi}(0)   (i summed): every start projected on every left eigenvector.   → (B, N)"""
    return torch.einsum("ik,bi->bk", left, p0)


def evolve(a: torch.Tensor, lam: torch.Tensor, right: torch.Tensor, times: torch.Tensor) -> torch.Tensor:
    r"""p_{bi}(t) = \sum_k r_{ik} a_{bk} e^{λ_k t}   (k summed, k = 1 is the stationary state).   → (T, B, N)"""
    decay = torch.exp(times[:, None] * lam[None, :])  # (T, N)  e^{λ_k t}
    return torch.einsum("ik,bk,tk->tbi", right, a, decay)


def kl_divergence(p: torch.Tensor, pi: torch.Tensor) -> torch.Tensor:
    r"""D(p ‖ π) = \sum_i p_i ln(p_i / π_i): distance to equilibrium (≥ 0, zero only at p = π).   (..., N) → (...)"""
    p = p.clamp_min(1e-300)
    return (p * (p.log() - pi.log())).sum(dim=-1).clamp_min(0.0)  # round-off can give -1e-12 once p ≈ π


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/thermodynamics/mpemba_effect/tensors.py
"""Build the concrete tensors: the lattice potential, the neighbour matrix and the batch of initial states."""

import math

import torch

# Staircase potential, left to right: a raised plateau, a deep narrow well, a barrier, a shallow wide well.
SEGMENTS = [(16, 1.0), (4, -3.0), (2, None), (18, -2.0)]  # (number of sites, energy); None → the barrier height


def potential(barrier: float, device: torch.device) -> torch.Tensor:
    """U_i on N = 40 lattice sites, shape (N,), in units of k_B (so U/T is dimensionless)."""
    parts = [torch.full((n,), barrier if e is None else e, dtype=torch.float64) for n, e in SEGMENTS]
    return torch.cat(parts).to(device)


def left_basin(n_sites: int, device: torch.device) -> torch.Tensor:
    """Boolean mask (N,) of the sites left of the barrier (plateau + deep well)."""
    n_left = SEGMENTS[0][0] + SEGMENTS[1][0]
    return torch.arange(n_sites, device=device) < n_left


def neighbours(n_sites: int, device: torch.device) -> torch.Tensor:
    """A_ij = 1 if sites i and j are nearest neighbours on the open chain, else 0: shape (N, N)."""
    i = torch.arange(n_sites, device=device)
    return ((i[:, None] - i[None, :]).abs() == 1).double()


def boltzmann(u: torch.Tensor, T: torch.Tensor) -> torch.Tensor:
    """p_bi = e^{-U_i / T_b} / Z_b for a batch of temperatures T (B,): shape (B, N). softmax keeps it finite."""
    return torch.softmax(-u[None, :] / T[:, None], dim=1)


def start_temperatures(T_bath: float, T_star: float, n: int, device: torch.device) -> torch.Tensor:
    """n initial temperatures from just above T_bath up to 30 T_bath (log spaced), plus the Mpemba point T*."""
    T = torch.logspace(math.log10(1.15 * T_bath), math.log10(30 * T_bath), n, dtype=torch.float64)
    T = torch.cat([T, torch.tensor([T_star], dtype=torch.float64)])
    return T.sort().values.to(device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/thermodynamics/mpemba_effect/solve.py
"""Diagonalise W once, then relax every initial temperature exactly; locate the strong Mpemba point."""

import torch

from . import tensors
from .equation import metropolis_rates, mode_amplitudes


def spectrum(W: torch.Tensor, pi: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Eigenvalues λ_k (0 = λ_1 > λ_2 > …) with right r_ik and left ℓ_ik eigenvectors of W, columns k.

    W is not symmetric, but detailed balance makes S = Π^{-1/2} W Π^{1/2} symmetric (Π = diag π), so the
    stable torch.linalg.eigh works: S v_k = λ_k v_k gives r_k = Π^{1/2} v_k and ℓ_k = Π^{-1/2} v_k, with ℓ_j·r_k = δ_jk.
    """
    s = pi.sqrt()  # (N,)
    S = W * s[None, :] / s[:, None]  # (N, N)  S_ij = W_ij √π_j / √π_i
    lam, V = torch.linalg.eigh(0.5 * (S + S.T))  # ascending; symmetrise away round-off
    lam, V = lam.flip(0), V.flip(1)  # slowest first: λ_1 = 0, λ_2 < 0, …
    V = V * torch.sign(V[0])[None, :]  # sign convention: every eigenvector positive on site 0
    right = s[:, None] * V  # (N, N)  r_ik
    left = V / s[:, None]  # (N, N)  ℓ_ik
    return lam, right, left


def a2_curve(u: torch.Tensor, T: torch.Tensor, left: torch.Tensor) -> torch.Tensor:
    """a_2(T) for a whole batch of initial temperatures in one einsum: (B,) → (B,)."""
    return mode_amplitudes(tensors.boltzmann(u, T), left)[:, 1]


def mpemba_point(u: torch.Tensor, T_bath: float, left: torch.Tensor, n_grid: int = 400) -> float:
    """Temperature T* > T_b where a_2 crosses zero: scan a log grid (batched), then refine by bisection."""
    T = torch.logspace(0.0, 2.0, n_grid, dtype=u.dtype, device=u.device) * 1.01 * T_bath  # (G,)
    a2 = a2_curve(u, T, left)
    flips = (torch.sign(a2[1:]) != torch.sign(a2[:-1])).nonzero()
    if len(flips) == 0:
        return float("nan")
    lo, hi = T[flips[0, 0]].item(), T[flips[0, 0] + 1].item()
    for _ in range(50):
        mid = 0.5 * (lo + hi)
        a_lo, a_mid = a2_curve(u, torch.tensor([lo, mid], dtype=u.dtype, device=u.device), left)
        lo, hi = (lo, mid) if torch.sign(a_lo) != torch.sign(a_mid) else (mid, hi)
    return 0.5 * (lo + hi)


def build(barrier: float, T_bath: float, device: torch.device) -> dict:
    """Potential, rate matrix, Boltzmann state and spectrum for one bath temperature."""
    u = tensors.potential(barrier, device)
    A = tensors.neighbours(len(u), device)
    W = metropolis_rates(u, A, T_bath)
    pi = tensors.boltzmann(u, torch.tensor([T_bath], dtype=u.dtype, device=device))[0]  # (N,)
    lam, right, left = spectrum(W, pi)
    return {"u": u, "W": W, "pi": pi, "lam": lam, "right": right, "left": left}


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/thermodynamics/mpemba_effect/plot.py
"""Top left: p_i(t) in the staircase potential for three starts. Top right: distance to equilibrium.
Bottom: the slow-mode amplitude a_2 against the initial temperature, with the strong Mpemba point."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.ticker import FixedLocator, FuncFormatter, NullLocator

from problems._common.plotting import frame_indices, save_gif

COLORS = {"warm": "#ea580c", "T*": "#dc2626", "hot": "#2563eb"}


def _plain_log_ticks(axis, lo: float, hi: float, every: int = 1) -> None:
    """Decade ticks with plain-text labels such as 1e3 (mathtext labels are slow to redraw in every GIF frame)."""
    decades = 10.0 ** np.arange(np.floor(np.log10(lo)), np.ceil(np.log10(hi)) + 1, every)
    axis.set_major_locator(FixedLocator(decades))
    axis.set_minor_locator(NullLocator())
    axis.set_major_formatter(FuncFormatter(lambda v, _: f"1e{np.log10(v):.0f}"))


def animate(
    u: torch.Tensor, pi: torch.Tensor, times: torch.Tensor, p: torch.Tensor, kl: torch.Tensor, T0: torch.Tensor,
    show: dict, T_grid: torch.Tensor, a2_grid: torch.Tensor, T_bath: float, T_star: float, path: Path,
) -> Path:  # fmt: skip
    us, pis, t, ps, kls = u.cpu().numpy(), pi.cpu().numpy(), times.cpu().numpy(), p.cpu().numpy(), kl.cpu().numpy()
    kls = np.maximum(kls, 1e-300)  # round-off can leave D ≈ -1e-12 once p = π
    T0s, Tg, a2g = T0.cpu().numpy(), T_grid.cpu().numpy(), a2_grid.cpu().numpy()
    x = np.arange(len(us))
    t_plot = np.maximum(t, t[1])  # log axis: draw t = 0 at the first positive time

    fig = plt.figure(figsize=(7.4, 4.8))
    grid = fig.add_gridspec(2, 2, height_ratios=[1.35, 1])
    ax_p, ax_k, ax_a = fig.add_subplot(grid[0, 0]), fig.add_subplot(grid[0, 1]), fig.add_subplot(grid[1, :])

    # potential (right axis) and the probability on each site
    ax_u = ax_p.twinx()
    ax_u.fill_between(x, us, us.min() - 0.5, step="mid", color="#e5e7eb", lw=0)
    ax_u.set_ylim(us.min() - 0.5, us.max() + 1.5)
    ax_u.set_ylabel("U (grey)", color="#6b7280", fontsize=8)
    ax_p.set_zorder(ax_u.get_zorder() + 1)  # probability curves above the grey potential
    ax_p.patch.set_visible(False)
    ax_u.grid(False)
    ax_p.plot(x, pis, color="#111827", lw=1, ls="--", label="π (Boltzmann, T_b)")
    lines = {k: ax_p.plot(x, ps[0, b], color=COLORS[k], lw=1.6, label=f"T₀ = {T0s[b]:.2f}")[0] for k, b in show.items()}
    ax_p.set_ylim(0, 1.15 * max(ps[:, list(show.values())].max(), pis.max()))
    ax_p.set_xlabel("site i")
    ax_p.set_ylabel("p_i(t)")
    ax_p.legend(fontsize=7, loc="upper right")
    title = ax_p.set_title(" ", fontsize=9)

    # D(p‖π) for every start, highlighted ones on top
    cmap = plt.get_cmap("Greys")
    for b in range(len(T0s)):
        if b not in show.values():
            ax_k.loglog(t_plot, kls[:, b], color=cmap(0.3 + 0.5 * b / len(T0s)), lw=0.8)
    for k, b in show.items():
        ax_k.loglog(t_plot, kls[:, b], color=COLORS[k], lw=1.8)
    y_lo = max(kls[-1].min(), 1e-12)
    ax_k.set_xlim(t_plot[0], t_plot[-1])
    ax_k.set_ylim(y_lo, kls.max() * 2)
    _plain_log_ticks(ax_k.xaxis, t_plot[0], t_plot[-1])
    _plain_log_ticks(ax_k.yaxis, y_lo, kls.max() * 2, every=3)
    ax_k.set_xlabel("time t")
    ax_k.set_ylabel("D(p ‖ π)")
    ax_k.set_title("distance to equilibrium (grey: other T₀)", fontsize=9)
    cursor = ax_k.axvline(t_plot[0], color="#f59e0b", lw=1.2)

    # a_2(T): a zero crossing above T_b is the strong Mpemba effect
    ax_a.semilogx(Tg, a2g, color="#374151", lw=1.6)
    ax_a.axhline(0, color="#9ca3af", lw=1)
    ax_a.axvline(T_bath, color="#9ca3af", ls="--", lw=1)
    ax_a.text(T_bath, a2g.max(), "T_b ", color="#6b7280", fontsize=8, va="top", ha="right")
    ax_a.axvline(T_star, color=COLORS["T*"], ls="--", lw=1)
    ax_a.text(T_star, a2g.max(), f" T* = {T_star:.2f}  (a₂ = 0)", color=COLORS["T*"], fontsize=8, va="top")
    a2_at = np.interp(np.log(T0s), np.log(Tg), a2g)
    for k, b in show.items():
        ax_a.plot([T0s[b]], [a2_at[b]], "o", color=COLORS[k], ms=6)
    ax_a.xaxis.set_major_locator(FixedLocator([1, 3, 10, 30, 100]))
    ax_a.xaxis.set_minor_locator(NullLocator())
    ax_a.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
    ax_a.set_xlabel("initial temperature T₀")
    ax_a.set_ylabel("a₂(T₀)")

    def update(step: int):
        for k, b in show.items():
            lines[k].set_ydata(ps[step, b])
        cursor.set_xdata([t_plot[step]])
        title.set_text(f"t = {t[step]:.3g}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 72), path, fps=12)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'T_bath': 1.0,
    'barrier': 2.0,
    'T_hot': 3.5
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.thermodynamics.mpemba_effect")]:
    del sys.modules[name]
main = importlib.import_module("problems.thermodynamics.mpemba_effect.main")
problem_dir = Path("problems/thermodynamics/mpemba_effect")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
